# 04 Git 版本控制与协作

> 覆盖范围：三区模型（工作区/暂存区/仓库）、常用命令、分支与合并（merge vs rebase）、冲突解决、撤销（reset/revert/checkout）、stash、远端协作（clone/pull/push/PR）、分支策略（Git Flow/Trunk-based）。
> 使用方式：先读**原理讲解**（三区流转、commit 链、refs），再看**命令示例**（默写），最后用 Python 模拟核心语义验证，自测面试追问。

---

## 1. Git 的本质：内容寻址的版本链

Git 不是「存文件差异」的简单工具，它的核心是两个数据结构：

1. **对象库（objects）**：所有内容（blob 文件内容、tree 目录树、commit 快照、tag）都被哈希（SHA-1）后存为不可变对象，内容即地址；
2. **引用（refs）**：`refs/heads/main` 等指向最新 commit 的指针；`HEAD` 指向当前分支。

**commit 链**：每个 commit 对象记录「父 commit 们 + 完整快照（根 tree）+ 作者/时间/消息」。所以：

- 一个 commit 的哈希由其**全部内容**决定——改了任何一字节，哈希就变（内容寻址保证完整性）；
- 历史就是**单向链表/树**：`HEAD -> main -> commit3 -> commit2 -> commit1`；
- 分支只是**可移动的指针**，标签（tag）是**不可移动**的指针。

## 2. 三区模型（最核心的直觉）

```
工作区(Working) --add--> 暂存区/索引(Staging/Index) --commit--> 仓库(Repository/HEAD)
```

| 区域 | 是什么 | 对应命令 |
| --- | --- | --- |
| **工作区** | 磁盘上你能编辑的文件 | `git status` 看变化 |
| **暂存区**（index） | 下次 commit 的内容快照（一个文件 `index`） | `git add` |
| **仓库**（HEAD） | 已提交的版本历史 | `git commit` |

**为什么要多一个暂存区**：把「修改」与「提交」解耦——你可以只挑部分文件（甚至部分行 `git add -p`）进入一次提交，形成逻辑清晰的提交历史。

## 3. 常用命令全景（默写级）

```bash
# 配置
git config --global user.name "你的名字"
git config --global user.email "you@example.com"

# 创建/克隆
git init                          # 当前目录初始化
git clone <url>                   # 克隆远端

# 三区流转
git status                        # 查看状态（-s 精简）
git add file / . / -p             # 加入暂存（-p 分块挑选）
git commit -m "msg"              # 提交（-am 跳过 add 只适用于已跟踪文件）

# 查看
git log --oneline --graph         # 简洁历史（画分支图）
git diff                          # 工作区 vs 暂存区
git diff --cached                 # 暂存区 vs HEAD
git show <hash>                   # 查看某次提交详情

# 分支
git branch                        # 列出分支
git branch <name>                 # 新建分支
git switch <name>                 # 切换分支（旧: checkout）
git switch -c <name>              # 新建并切换
git merge <branch>                # 合并分支到当前

# 撤销
git restore <file>                # 丢弃工作区修改（旧: checkout -- file）
git restore --staged <file>       # 取消暂存（旧: reset HEAD file）
git reset --soft HEAD~1           # 撤销 commit，保留暂存
git reset --mixed HEAD~1          # 撤销 commit 和暂存，保留工作区（默认）
git reset --hard HEAD~1           # 全部丢弃（⚠️ 危险）
git revert <hash>                 # 新增一个反向提交（安全撤销已推送历史）
git stash                         # 暂存工作区（stash list/pop 恢复）

# 远端
git remote -v                     # 查看远端
git fetch                         # 拉取远端历史（不合并）
git pull                          # fetch + merge
git push                          # 推送到远端
git push -u origin main           # 首次推送并建立跟踪
```

## 4. merge vs rebase：两种合入方式

假设 main 在 A，feature 分支从 A 分出做到 D：

```
      A---B---C   main
       \
        D---E      feature
```

### 4.1 merge（保持历史真实）

```
git switch main && git merge feature
      A---B---C---M   main
       \         /
        D---E        feature
```

- 产生一个**合并提交 M**，两条历史线保留；
- 不重写历史，**适合共享分支**（main）与团队协作；
- 历史会呈现分支分叉，较复杂时 `--graph` 可见分叉。

### 4.2 rebase（线性重放）

```
git switch feature && git rebase main
      A---B---C        main
               \
                D'--E'  feature（D、E 被重放到 C 之后，哈希变了）
```

- 把 feature 的提交**摘下来重放到 main 顶**，历史**线性**；
- **重写了提交哈希**——已推送共享的分支**不要 rebase**（别人会冲突）；
- 适合本地未推送的分支整理（squash 合并多个提交、修改提交信息）。

> **面试一句话**：merge 保留真实历史（有合并节点），rebase 线性化历史（重写哈希）；共享分支用 merge，私有分支整理用 rebase。

## 5. 冲突解决

双方修改同一处时合并产生冲突：

```text
<<<<<<< HEAD
当前分支的版本
=======
合并进来的版本
>>>>>>> feature
```

解决步骤：

1. `git status` 找到冲突文件（`both modified`）；
2. 编辑文件，保留想要的版本，**删除 `<<<<<<<` / `=======` / `>>>>>>>` 标记**；
3. `git add <file>` 标记已解决；
4. `git commit` 完成合并（或 rebase 时 `git rebase --continue`）。

## 6. reset vs revert（撤销的两种哲学）

| 操作 | 做什么 | 是否改写历史 | 适用 |
| --- | --- | --- | --- |
| `git reset` | 移动分支指针**回退**，之后的提交消失（可用 reflog 找回） | 是 | 本地未推送的提交 |
| `git revert` | **新增一个反向提交**抵消目标提交 | 否 | 已推送到远端的历史 |

**reflog 救命**：`git reflog` 记录所有指针移动，`git reset --hard <hash>` 找回误删的提交。

## 7. 分支策略

| 策略 | 模式 | 适合 |
| --- | --- | --- |
| **Git Flow** | main（发布）+ develop（开发）+ feature/ + release/ + hotfix/ | 版本化发布的产品（历史项目） |
| **Trunk-Based** | 主干 short-lived 分支 + 频繁合并/PR | 持续交付/CI CD 快的团队（主流） |
| **GitHub Flow** | main 永远可发布，feature 分支 + PR | 互联网产品团队 |

**PR（Pull Request）/ MR**：不是 Git 特性，是平台（GitHub/GitLab）的**代码评审流程**：push feature → 提 PR → CI 检查 + 人审 → merge。


## 核心概念

### (a) 🗂️ Git 从「版本链」到「团队协作」的主线

这篇从 Git 的底层数据结构（内容寻址的 commit 链）讲起，然后是每天都要用的三区模型与命令，再到分支、merge/rebase、冲突解决、撤销回滚和远端协作——理解和记忆都建立在「分支是指针」这个直觉上。

```mermaid
flowchart TB
    Node1["🧠 Git：内容寻址的版本链"] --> Node2["🗂️ 三区模型"]
    Node2 --> Node3["✍️ 日常流转：add / commit / status"]
    Node3 --> Node4["🌿 分支：可移动指针 + HEAD"]
    Node4 --> Node5["🔀 merge：保留真实历史"]
    Node4 --> Node6["📏 rebase：线性重放"]
    Node5 --> Node7["⚔️ 冲突与解决"]
    Node1 --> Node8["⏪ 撤销：reset vs revert + reflog"]
    Node8 --> Node9["🌐 远端协作：clone / pull / push / PR"]
```

- **🧠 Git：内容寻址的版本链**：Git 不是「存差异」的工具：文件内容（blob）、目录树（tree）、提交（commit）都被 SHA-1 哈希后存成不可变对象，内容即地址；每个 commit 记录父提交 + 完整快照 + 作者信息，历史就是一条单向链表。
- **🗂️ 三区模型**：工作区是你正在编辑的磁盘文件；暂存区（index）是「下次 commit 的内容快照」；仓库（HEAD）是已提交的历史。多一个暂存区是为了把「修改」和「提交」解耦——可以只挑部分文件、甚至部分行进一次提交。
- **✍️ 日常流转**：`git status` 看状态、`git add` 把文件放进暂存区、`git commit` 生成快照、`git diff` 比较工作区与暂存区、`git diff --cached` 比较暂存区与 HEAD。这些是每天用得最多的命令。
- **🌿 分支**：分支的本质只是指向某个 commit 的**可移动指针**，tag 是不可移动指针，`HEAD` 指向当前分支——所以建分支零成本，可以随便开。
- **🔀 merge**：合并时产生一个合并提交 M，两条历史线都保留，不重写历史。适合共享分支（main）和团队协作，缺点是多出分叉、历史不够线性。
- **📏 rebase**：把 feature 的提交摘下来**重放**到 main 的顶端，历史变成一条直线，但**重写了提交哈希**——已经推送的共享分支别 rebase（别人会冲突）；本地未推送的分支用它整理（squash 合并、改提交信息）。
- **⚔️ 冲突与解决**：两分支改了同一处就会冲突，文件里出现 `<<<<<<< HEAD ... ======= ... >>>>>>>` 标记。解决：编辑保留想要的版本、删掉标记 → `git add` → `git commit`（或 rebase 时 `--continue`）。
- **⏪ 撤销**：`reset` 移动分支指针回退、之后的提交消失（改写历史，适合本地未推送）；`revert` 新增一个反向提交抵消目标提交（不改写历史，适合已推送远端）；误删了用 `git reflog` 找回。
- **🌐 远端协作**：`clone` 克隆、`fetch` 只拉历史不合并、`pull = fetch + merge`、`push` 推送。PR 不是 Git 特性而是平台（GitHub/GitLab）的代码评审流程：push 分支 → 提 PR → CI 检查 + 人审 → merge；分支策略主流是 Git Flow（版本化发布）或 Trunk-Based（持续交付）。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 模拟 Git 三区模型：工作区 -> 暂存区 -> 仓库（commit 链）

import hashlib

class MiniGit:
    def __init__(self):
        self.working = {}        # 工作区文件
        self.index = {}          # 暂存区（index）
        self.commits = []        # commit 链（每个是 (msg, snapshot, parent)）
        self.head = None         # HEAD 指向最新 commit 索引

    def edit(self, path, content):
        self.working[path] = content

    def add(self, path):
        # git add：把工作区内容快照进暂存区
        self.index[path] = self.working[path]

    def commit(self, msg):
        # git commit：把暂存区快照固化为 commit，HEAD 前进
        snapshot = dict(self.index)
        parent = self.head
        digest = hashlib.sha1((msg + str(snapshot)).encode()).hexdigest()[:8]
        self.commits.append({'msg': msg, 'snap': snapshot, 'parent': parent, 'hash': digest})
        self.head = len(self.commits) - 1
        return digest

    def log(self):
        # git log --oneline：沿 parent 链回溯
        out = []
        i = self.head
        while i is not None:
            c = self.commits[i]
            out.append(f'{c["hash"]} {c["msg"]}')
            i = c['parent']
        return out

    def show(self, rev):
        return self.commits[rev]['snap']

g = MiniGit()
g.edit('main.py', 'print(1)')
g.add('main.py')
h1 = g.commit('feat: 初始版本')
g.edit('main.py', 'print(2)')
g.edit('README.md', 'docs')
g.add('README.md')                      # 只暂存 README
h2 = g.commit('docs: README')
g.add('main.py')
h3 = g.commit('feat: 打印2')
print('commit 链:')
for line in g.log(): print(' ', line)
print('\ncommit2 快照:', g.show(1))
print('commit3 快照:', g.show(2))
print('\n验证：只有 add 过的文件才进入提交（暂存区的作用）')
assert g.show(2)['main.py'] == 'print(2)'

In [ ]:
# 模拟分支 + fast-forward / 三路合并（merge）

class Branch:
    def __init__(self, name, head):
        self.name, self.head = name, head

class Repo:
    def __init__(self):
        self.commits = {}
        self.branches = {}
        self.head = None

    def commit(self, branch, msg):
        parent = self.branches[branch].head
        sha = f'{branch}-{len(self.commits)}'
        self.commits[sha] = {'msg': msg, 'parent': parent}
        self.branches[branch].head = sha
        return sha

r = Repo()
r.branches['main'] = Branch('main', None)
r.commit('main', 'A')          # A
r.commit('main', 'B')          # A->B
r.branches['feature'] = Branch('feature', r.branches['main'].head)  # feature 从 B 分出
r.commit('feature', 'D')       # B->D
r.commit('main', 'C')          # main: B->C（与 feature 分叉）
r.commit('feature', 'E')       # feature: D->E

print('main   :', r.branches['main'].head)
print('feature:', r.branches['feature'].head)

# 模拟 merge：找到共同祖先 B
def find_merge_base(c1, c2):
    """找两个提交的最近共同祖先（朴素实现）"""
    anc = set()
    while c1:
        anc.add(c1)
        c1 = r.commits[c1]['parent']
    while c2:
        if c2 in anc:
            return c2
        c2 = r.commits[c2]['parent']

base = find_merge_base(r.branches['main'].head, r.branches['feature'].head)
print('共同祖先:', base, '<- 三路合并以它为基准比较')
if base == r.branches['feature'].head:
    print('fast-forward：feature 无新提交，main 直接前移')
else:
    print('产生合并提交 M（main <- feature 的独有提交并入）')
assert base == 'main-1'   # B（main 的第二次提交）是分叉点

In [ ]:
# 模拟三路合并算法本体：base + ours + theirs -> merged（理解冲突来源）

def three_way_merge(base, ours, theirs):
    """返回 (合并结果, 冲突列表)。
    规则：一方改了就用那一方；两方都改且不同 -> 冲突。
    """
    result = {}
    conflicts = []
    keys = set(base) | set(ours) | set(theirs)
    for k in sorted(keys):
        b, o, t = base.get(k), ours.get(k), theirs.get(k)
        if o == t:
            result[k] = o           # 都改或都没改且一致
        elif b == o:
            result[k] = t           # 只有对方改 -> 用对方的
        elif b == t:
            result[k] = o           # 只有我们改 -> 用我们的
        else:
            conflicts.append(k)     # 双方都改了不同内容 -> 冲突
    return result, conflicts

# 场景：base 版本 config.py
base = {'host': '127.0.0.1', 'port': 8000, 'debug': False}
# main 分支改了 port
ours = {'host': '127.0.0.1', 'port': 9000, 'debug': False}
# feature 分支改了 host，也改了 port（与 main 不同）
theirs = {'host': '0.0.0.0', 'port': 8080, 'debug': False}

merged, conflicts = three_way_merge(base, ours, theirs)
print('合并结果:', merged)
print('冲突字段:', conflicts, '<- port 双方都改且不同，需要人工解决')
assert 'port' in conflicts and merged['host'] == '0.0.0.0'
print('\n冲突文件标记示例（git 会写成这样）：')
print('<<<<<<< HEAD')
print('port = 9000   # ours')
print('=======')
print('port = 8080   # theirs')
print('>>>>>>> feature')

## 8. 面试 30 秒速答

- **三区**：工作区 → `add` → 暂存区 → `commit` → 仓库；暂存区让「修改」与「提交」解耦；
- **commit 链**：commit = 父子指针 + 完整快照 + 内容哈希；分支是指针，tag 是不可动指针；
- **merge vs rebase**：merge 产生合并节点、保留真实历史；rebase 重放提交让历史线性但**重写哈希**——共享分支别 rebase；
- **冲突**：两分支改同一处；手动编辑删除 `<<<<<<< ======= >>>>>>>` 标记后 add+commit；
- **reset vs revert**：reset 回退指针（改写历史，本地用）；revert 新增反向提交（不改写，远端用）；reflog 可找回误删；
- **stash**：临时保存未提交改动，pop 恢复；
- **pull** = fetch + merge；push 前先 pull 避免冲突。

## 9. 自测清单（面试追问）

- [ ] 三区模型各是什么？为什么需要暂存区？
- [ ] commit 对象里存了什么？为什么哈希能保证完整性？
- [ ] 分支的本质是什么？HEAD 指向什么？tag 和 branch 区别？
- [ ] merge 和 rebase 的区别？什么时候用哪个？rebase 为什么危险？
- [ ] 手写：冲突标记长什么样？解决冲突的完整命令序列？
- [ ] reset 的 soft/mixed/hard 区别？revert 与 reset 怎么选？
- [ ] 误删了重要提交怎么找回？（reflog）
- [ ] `git pull` 与 `git fetch` 区别？push 被拒绝（远端有新提交）怎么办？
- [ ] `git add -p` 是干什么的？stash 有什么用？
- [ ] Git Flow / Trunk-Based 各适合什么团队？Code Review 流程（PR）是怎样的？
- [ ] detached HEAD 是什么？怎么发生的？（checkout 到具体 commit）
- [ ] 手写：把 3 个本地提交合并成 1 个（rebase -i squash）。

> ✅ 全部能回答 + 能默写命令，本课通过。